# finetune_v2_run — fine-tune với bằng chứng dương SẠCH, đo đầy đủ trên dev300

**~5h, MỘT lượt.** GPU T4, Internet On, **Save & Run All (Commit)**.

## Vì sao thử lại, dù 13/08 đã thua

CLAUDE.md cấm "thử lại bằng cách vặn siêu tham số". Lượt này **không vặn gì**:
`epochs=1`, `lr=2e-5`, `n_neg=4`, `bs=8`, `max_length=512` — y hệt 13/08. Đổi đúng
**một thứ**: đoạn văn bản dùng làm ví dụ huấn luyện.

`finetune_rerank.best_chunk` bản cũ dùng `max(parts, key=len(qs & tok(c)))` — đếm từ
trùng thô trên đoạn `chunks_of` không gộp. **Đúng hàm mà 17/08 đo được là kém BM25
mức đoạn 1,34 điểm.** Hậu quả đo được trên 24 văn bản gold khó nhất của dev300:

| băm | hàm chọn | positive trùng đoạn CE thích nhất |
|---|---|---|
| `chunks_of` thô | `count` ← **bản 13/08** | **3/24 = 12,5%** |
| `chunks_of` thô | `bm25` | 5/24 |
| gộp 1800 | `count` | 6/24 |
| **gộp 1800** | **`bm25`** ← bản này | **8/24 = 33%** |

Gần 9/10 ví dụ dương của lượt 13/08 là **bằng chứng sai**. Với nhóm câu mà CE chấm
~0,00 trên toàn văn bản gold, positive là dòng **mục lục** không chứa câu trả lời —
model bị dạy "dòng mục lục này trả lời câu hỏi này".

Tầng thứ hai: **train lệch inference**. Lúc chấm model thấy đoạn BM25 gộp ~1.460 ký tự;
lúc huấn luyện nó thấy đoạn term-overlap ~650 ký tự. Nó chưa từng thấy loại đầu vào
phải chấm. Bản vá gọi thẳng `DC.pick_chunks` — **cùng một hàm**, không chép lại, vì
chép là hai bên trôi khỏi nhau, mà trôi khỏi nhau chính là bug này.

## Đọc kết quả — Bước 4 in thẳng

So với **0.9183** (model gốc, cùng cấu hình BM25+gộp1800, M=20 K=20, n=2):

| ft ra | quyết định |
|---|---|
| **≥ 0.9383** (+2,0) | thắng rõ → chạy đề thi 2 lượt, nộp |
| 0.9183 – 0.9383 | dev300 **không phân giải nổi** → ĐỪNG đốt 10h đề thi. Xem mục dưới |
| < 0.9183 | fine-tune thua lần hai → **đóng hướng này vĩnh viễn** |

Ngưỡng +2,0 là quy tắc 4: dev300 có 300 câu, 1,05 gold/câu, nhiễu ±1,5 câu. Bài học
18/08 mới toanh: dev300 báo +1,00 cho BM25+gộp, public LB trả về **+0,31** — nó chưa
bao giờ *đo* được hiệu ứng dưới 2 điểm, chỉ tung đồng xu.

## Upload lên dataset `project-ir`

| file | ghi chú |
|---|---|
| `finetune_rerank.py` | **BẢN MỚI**, đè bản cũ |
| `deep_chunk.py` | bản BM25 (`sha256:2a8c193a1e4a`). Đã upload cho lượt trước thì thôi |

`train.json`, `dev_1000_locked.json`, `dev_300_locked.json`, `bm25_top100_dev300.json`,
`selected-contexts/`, file hard-negative — đã có sẵn.

In [ ]:
!pip install -q sentence-transformers

import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Kaggle đặt dataset ở một trong hai chỗ tuỳ cách gắn. Tự dò thay vì đoán.
INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
sys.path.append(INPUT_DIR)
print("INPUT_DIR =", INPUT_DIR)
!ls {INPUT_DIR}


In [ ]:
import json, time, hashlib, inspect, subprocess
from pathlib import Path

from rerank import load_reranker
from rerank_from_d import score_all_from_d, blend_bm25_first
import deep_chunk as DC
import finetune_rerank as FT

# Kaggle giải nén zip -> corpus có thể nằm LỒNG một lớp. Chọn thư mục có NHIỀU
# file context_*.json NHẤT — đừng lấy cái đầu tiên "có vẻ đúng", và đừng đếm
# entry: 19/08 thư mục ngoài có 8.532 entry nhưng chỉ chứa thư mục con.
def _n_ctx(p):
    return (sum(1 for f in os.listdir(p)
                if f.startswith("context_") and f.endswith(".json"))
            if os.path.isdir(p) else -1)

CTX = max((f"{INPUT_DIR}/selected-contexts/selected-contexts",
           f"{INPUT_DIR}/selected-contexts"), key=_n_ctx)
corpus_ids = {f[8:-5] for f in os.listdir(CTX)
              if f.startswith("context_") and f.endswith(".json")}
print(f"CTX = {CTX}")
print(f"      {len(os.listdir(CTX)):,} entry | {len(corpus_ids):,} file context_*.json")
assert len(corpus_ids) == 8532, (
    f"corpus chỉ {len(corpus_ids):,}/8.532 văn bản — CTX trỏ sai chỗ")

MERGE_CHARS, M_DOC, K_CHUNK, TOPK = 1800, 20, 20, 5
MOC_GOC = 0.9183          # model GỐC, cùng cấu hình. Đây là số phải vượt
NGUONG  = MOC_GOC + 0.02  # quy tắc 4: dưới +2,0 thì dev300 không phân giải được
OUT = "/kaggle/working/outputs"; Path(OUT).mkdir(parents=True, exist_ok=True)
FT_DIR = "/kaggle/working/ft_model_v2"

DC.MERGE_CHARS = MERGE_CHARS
for m in (DC, FT):
    f = inspect.getsourcefile(m); b = open(f, "rb").read()
    print(f"  {os.path.basename(f):24s} {len(b):>7,}B  sha256:{hashlib.sha256(b).hexdigest()[:12]}")

# Kiểm bằng CHỮ KÝ HÀM, không bằng text: docstring bản mới có TRÍCH DẪN code cũ nên
# `"len(qs & set(tok(c)))" not in src` tự bắn vào chân mình (đã dính 19/08).
sig = list(inspect.signature(FT.best_chunk).parameters)
assert sig == ["question", "ctx_dir", "doc_id"], (
    f"finetune_rerank.py là BẢN CŨ — best_chunk{tuple(sig)}, phải là "
    "(question, ctx_dir, doc_id). Upload đè lên dataset.")
assert "DC.pick_chunks" in inspect.getsource(FT.best_chunk), "best_chunk không gọi DC.pick_chunks"
assert "idf.get" in inspect.getsource(DC.pick_chunks), "deep_chunk.py là bản cũ"
r = subprocess.run([sys.executable, inspect.getsourcefile(DC)], capture_output=True, text=True)
print(f"  self-test deep_chunk: {r.stdout.strip()}"); assert r.returncode == 0
print(f"\n  best_chunk = DC.pick_chunks BM25  OK | MERGE_CHARS={DC.MERGE_CHARS}")

## Bước 1 — Dựng dữ liệu huấn luyện

Băm + dựng chỉ mục BM25 cho ~7.000 văn bản → **chậm hơn 13/08 (449s)**, ước 15-25 phút.
`_cache` phình tới ~9GB rồi được xoá ngay sau đó, trước khi nạp model.

Chốt chặn: `hard negative` phải ra **24.000/24.000**. Ra `0/24,000` là `HARD_NEG` trỏ
sai chỗ — dừng ngay, đừng phí 3 tiếng (bẫy đã dính 12/08).

In [ ]:
CANDS = None
for name in ("bm25_ids_train_FALLBACK.json", "Input/bm25_ids_train_FALLBACK.json",
             "bm25_top100_train.json", "candidates_public_FB.json"):
    if os.path.isfile(f"{INPUT_DIR}/{name}"):
        CANDS = f"{INPUT_DIR}/{name}"; break
print(f"hard negative: {CANDS or 'KHÔNG THẤY -> dùng negative NGẪU NHIÊN (yếu hơn, lệch 13/08)'}")

TRAIN = next(p for p in (f"{INPUT_DIR}/Input/train.json", f"{INPUT_DIR}/train.json")
             if os.path.isfile(p))
train = json.load(open(TRAIN, encoding="utf-8"))
dev_qids = set(json.load(open(f"{INPUT_DIR}/dev_1000_locked.json", encoding="utf-8")))
cands = json.load(open(CANDS, encoding="utf-8")) if CANDS else None
if cands:
    v = next(iter(cands.values()))
    if v and not isinstance(v[0], dict):            # file chỉ có doc_id trần
        cands = {q: [{"doc_id": str(x)} for x in c] for q, c in cands.items()}
        print("  (đã bọc doc_id trần thành {'doc_id': ...})")

# CTX và corpus_ids lấy từ cell 2 — ĐỪNG định nghĩa lại ở đây. 19/08 dòng
# `CTX = f"{INPUT_DIR}/selected-contexts"` ở cell này ghi đè đường dẫn đã dò
# đúng, build_pairs nhận corpus rỗng và chết ở doc_id đầu tiên.
hoc = [q for q in train if q not in dev_qids]
thieu = {str(g) for q in hoc for g in train[q]["answer"]} - corpus_ids
assert not thieu, (f"{len(thieu)} gold doc_id KHÔNG có trong corpus "
                   f"(vd {sorted(thieu)[:5]}) — CTX trỏ sai chỗ")
print(f"{len(hoc):,} câu huấn luyện | {len(corpus_ids):,} văn bản | mọi gold đều có  OK")

t0 = time.time()
pairs = FT.build_pairs(train, dev_qids, corpus_ids, CTX, cands, n_neg=4)
print(f"dựng xong trong {(time.time()-t0)/60:.0f} phút | {len(pairs):,} cặp")
assert not (dev_qids & {q for q in train if q not in dev_qids}), "RÒ RỈ DEV"

npos = sum(1 for _, _, y in pairs if y == 1)
dai = sum(len(d) for _, d, _ in pairs) / len(pairs)
print(f"positive {npos:,} | negative {len(pairs)-npos:,} | đoạn dài TB {dai:.0f} ký tự")
assert dai > 900, f"đoạn TB {dai:.0f} ký tự — quá ngắn, MERGE_CHARS chưa ăn (13/08 là ~650)"
q, d, y = next(p for p in pairs if p[2] == 1)
print(f"\nví dụ positive:\n  Q: {q[:90]}\n  D: {d[:200]}")

## Bước 2 — Huấn luyện (~20-50 phút)

`bs=8` + AMP fp16 — `bs=16` fp32 là OOM trên T4 16GB (đã dính 12/08). Dòng `step 100`
in VRAM đỉnh; qua được mốc đó là an toàn tới hết.

In [ ]:
t0 = time.time()
FT.train_model(pairs, "AITeamVN/Vietnamese_Reranker", FT_DIR,
               epochs=1, bs=8, lr=2e-5, max_length=512, device="cuda")
print(f"train xong trong {(time.time()-t0)/60:.0f} phút")
del pairs

## Bước 3 — Đo trên dev300, ĐẦY ĐỦ hai tầng (~3,5h)

**Không dùng cổng chặn tầng-1-thôi.** Tầng 1 chấm `top_chunks` của D (~1.387 ký tự,
chọn kiểu khác) — đó đúng là loại đầu vào model này KHÔNG được huấn luyện, nên đo ở
đó sẽ đánh giá thấp nó. Phải đo ở tầng 2, nơi đầu vào khớp dữ liệu huấn luyện.

Lưu `scores` ngay sau tầng 1 (quy tắc 2) — tầng 2 hỏng thì lượt GPU vẫn còn tài sản.

In [ ]:
DC._cache.clear()                                   # cache của build_pairs, ~9GB
dev  = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))
cand = json.load(open(f"{INPUT_DIR}/bm25_top100_dev300.json", encoding="utf-8-sig"))
dev_q = {q: v["question"] for q, v in dev.items()}
gold  = {q: {str(x) for x in v["answer"]} for q, v in dev.items()}
bm25  = {q: [str(c["doc_id"]) for c in cand[q]] for q in dev_q}

score_fn = load_reranker(FT_DIR, device="cuda")

t0 = time.time()
scores = score_all_from_d(dev_q, cand, score_fn)
json.dump(scores, open(f"{OUT}/scores_dev300_ftv2_tang1.json", "w", encoding="utf-8"),
          ensure_ascii=False)
print(f"tầng 1 xong {(time.time()-t0)/60:.0f} phút — ĐÃ LƯU")

n2 = DC.count_deep_chunks(dev_q, scores, CTX, M_DOC, K_CHUNK)
print(f"tầng 2: {n2:,} đoạn (mong đợi ~108.000) | ước {n2/10.0/3600:.1f}h")
assert 80_000 < n2 < 140_000, f"{n2:,} lệch xa — kiểm M_DOC/K_CHUNK/MERGE_CHARS"

t0 = time.time()
deep = DC.deepen_all(dev_q, scores, CTX, score_fn, M_DOC, K_CHUNK, every=50)
print(f"tầng 2 xong {(time.time()-t0)/60:.0f} phút | nhịp {n2/(time.time()-t0):.1f} đoạn/s")
json.dump(deep, open(f"{OUT}/scores_dev300_ftv2_deep_M20_K20.json", "w", encoding="utf-8"),
          ensure_ascii=False)

## Bước 4 — Bảng kết quả và QUYẾT ĐỊNH

`n_bm25` quét lại vì điểm đổi thì đỉnh có thể dời. **Nhưng đừng đổi khỏi n=2 dựa vào
dev300** — 18/08 dev300 báo n=0 hơn n=2, public LB trả lời ngược lại (0.8871 vs 0.8851).

In [ ]:
def rec(S, variant, n):
    pred = {q: blend_bm25_first(DC.rank_by(S[q], variant), bm25[q], k=TOPK, n_bm25=n)
            for q in dev_q}
    return sum(len(gold[q] & set(pred[q])) / len(gold[q]) for q in gold) / len(gold)

tab = {v: {n: rec(deep, v, n) for n in (0, 1, 2, 3)} for v in DC.VARIANTS}
print(f"{'biến thể':10s}" + "".join(f"  n={n}    " for n in (0, 1, 2, 3)))
for v, row in tab.items():
    print(f"{v:10s}" + "".join(f"  {row[n]:.4f} " for n in (0, 1, 2, 3)))
print("\nmodel GỐC cùng cấu hình:  base 0.8883 (n=2) | max 0.9217/0.9117/0.9183/0.9100")

got = tab["max"][2]
print(f"\nft  max n=2 = {got:.4f}  | gốc {MOC_GOC}  | Δ {(got-MOC_GOC)*100:+.2f} điểm")
print("=" * 68)
if got >= NGUONG:
    print(f"THẮNG RÕ (+{(got-MOC_GOC)*100:.2f} ≥ 2,0). Chạy final_v2_run.ipynb hai lượt với")
    print(f"RERANKER_MODEL = ft_model_v2, rồi nộp. NHỚ TẢI ft_model_v2 VỀ MÁY.")
elif got >= MOC_GOC:
    print(f"VÙNG MÙ (+{(got-MOC_GOC)*100:.2f} < 2,0). dev300 KHÔNG phân giải được mức này —")
    print("18/08 nó báo +1,00 mà public chỉ trả +0,31. ĐỪNG đốt 10h đề thi cho con số này.")
else:
    print(f"THUA ({(got-MOC_GOC)*100:+.2f}). Fine-tune thua lần thứ hai, lần này với dữ liệu")
    print("đã sạch. ĐÓNG HƯỚNG NÀY VĨNH VIỄN — ghi vào CLAUDE.md, đừng ai thử lại.")
print("=" * 68)

json.dump({"table": tab, "moc_goc": MOC_GOC, "max_n2": got, "delta": got - MOC_GOC,
           "MERGE_CHARS": MERGE_CHARS, "M_DOC": M_DOC, "K_CHUNK": K_CHUNK,
           "hard_neg": CANDS, "n_pairs": len(json.load(open(f"{OUT}/scores_dev300_ftv2_tang1.json", encoding="utf-8")))},
          open(f"{OUT}/ftv2_eval.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

import shutil
shutil.make_archive(f"{OUT}/ft_model_v2", "zip", FT_DIR)
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  {os.path.getsize(os.path.join(OUT, f)):,} bytes")
print("\nTẢI TOÀN BỘ outputs/ — nhất là ft_model_v2.zip (không có nó phải train lại)")